In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from Bio import Phylo, SeqIO
from scipy import stats
from scipy.linalg import cho_factor, cho_solve
from scipy.optimize import minimize_scalar

OUT = Path("results/temperature")
OUT.mkdir(parents=True, exist_ok=True)
PRIMARY = ["proline_fraction", "gravy", "arg_over_arg_plus_lys"]
KD = {
    "A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5, "Q": -3.5, "E": -3.5,
    "G": -0.4, "H": -3.2, "I": 4.5, "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8,
    "P": -1.6, "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2,
}


def tokens(text):
    cleaned = str(text).lower().replace("sp.", " ").replace("_", " ").replace(".", "-")
    cleaned = re.sub(r"[^a-z0-9 -]+", " ", cleaned)
    return {token for token in cleaned.split() if any(character.isdigit() for character in token) and len(token) >= 3}


def holm(pvalues):
    values = np.asarray(pvalues, dtype=float)
    order = np.argsort(values)
    adjusted = np.empty(len(values))
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(values) - rank) * values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted.tolist()


def prune_to(tree, keep):
    for tip in list(tree.get_terminals()):
        if tip.name in keep:
            continue
        if tip not in tree.get_terminals():
            continue
        tree.prune(tip)
    changed = True
    while changed:
        changed = False
        for clade in list(tree.find_clades()):
            if clade is tree.root or clade.is_terminal() or len(clade.clades) != 1:
                continue
            child = clade.clades[0]
            child.branch_length = float(child.branch_length or 0) + float(clade.branch_length or 0)
            parent = next(candidate for candidate in tree.find_clades() if clade in candidate.clades)
            parent.clades[parent.clades.index(clade)] = child
            changed = True
            break
    return tree


def annotate_depth(clade, depth):
    clade.depth_from_root = depth
    for child in clade.clades:
        if child.branch_length is None or child.branch_length < 0:
            raise SystemExit("missing or negative branch length")
        annotate_depth(child, depth + float(child.branch_length))


def covariance(tree):
    annotate_depth(tree.root, 0.0)
    tips = list(tree.get_terminals())
    names = [tip.name for tip in tips]
    matrix = np.zeros((len(tips), len(tips)))
    for i, left in enumerate(tips):
        for j, right in enumerate(tips):
            if i == j:
                matrix[i, j] = left.depth_from_root
            else:
                matrix[i, j] = tree.common_ancestor(left, right).depth_from_root
    return matrix, names


def lambda_matrix(base, lam):
    diagonal = np.diag(np.diag(base))
    return lam * base + (1.0 - lam) * diagonal


def gls(y, predictor, base, lam):
    n = len(y)
    design = np.ones((n, 1)) if predictor is None else np.column_stack([np.ones(n), predictor])
    covariance_matrix = lambda_matrix(base, lam)
    factor = cho_factor(covariance_matrix, lower=True, check_finite=False)
    logdet = 2.0 * np.sum(np.log(np.diag(factor[0])))
    xt_vinv = cho_solve(factor, design).T
    xt_vinv_x = design.T @ xt_vinv.T
    beta = np.linalg.solve(xt_vinv_x, xt_vinv @ y)
    residual = y - design @ beta
    sigma2 = float(residual @ cho_solve(factor, residual)) / (n - design.shape[1])
    beta_var = sigma2 * np.linalg.inv(xt_vinv_x)
    df = n - design.shape[1]
    loglik = -0.5 * (df * np.log(2 * np.pi) + df * np.log(sigma2) + logdet + np.linalg.slogdet(xt_vinv_x)[1] + df)
    white = np.linalg.cholesky(covariance_matrix)
    standardized = np.linalg.solve(white, residual) / np.sqrt(sigma2)
    return {
        "beta": beta,
        "se": np.sqrt(np.diag(beta_var)),
        "sigma2": sigma2,
        "df": df,
        "loglik": float(loglik),
        "standardized_residual": standardized,
        "condition": float(np.linalg.cond(covariance_matrix)),
    }


def fit_lambda(y, predictor, base):
    def objective(lam):
        return -gls(y, predictor, base, float(lam))["loglik"]

    best = float(minimize_scalar(objective, bounds=(0.0, 1.0), method="bounded", options={"xatol": 1e-5}).x)
    result = gls(y, predictor, base, best)
    slope = np.nan if predictor is None else float(result["beta"][1])
    se = np.nan if predictor is None else float(result["se"][1])
    if predictor is None:
        pvalue = low = high = np.nan
    else:
        pvalue = float(2 * stats.t.sf(abs(slope / se), result["df"]))
        critical = float(stats.t.ppf(0.975, result["df"]))
        low = slope - critical * se
        high = slope + critical * se
    profile = np.linspace(0.0, 1.0, 101)
    profile_lik = np.array([gls(y, predictor, base, float(value))["loglik"] for value in profile])
    inside = profile[profile_lik >= profile_lik.max() - 1.920729]
    return {
        "lambda": best,
        "lambda_ci_low": float(inside.min()) if len(inside) else np.nan,
        "lambda_ci_high": float(inside.max()) if len(inside) else np.nan,
        "slope": slope,
        "se": se,
        "ci_low": low,
        "ci_high": high,
        "p": pvalue,
        "df": int(result["df"]),
        "sigma2": float(result["sigma2"]),
        "loglik": result["loglik"],
        "condition": result["condition"],
        "n": int(len(y)),
    }


def effective_n(base, lam):
    matrix = lambda_matrix(base, lam)
    scale = np.sqrt(np.diag(matrix))
    correlation = matrix / np.outer(scale, scale)
    precision = np.linalg.inv(correlation)
    ones = np.ones(len(base))
    return float(ones @ precision @ ones)


records = list(SeqIO.parse("results/alignment/colap_trimmed_aligned.fasta", "fasta"))
reference = next(record for record in records if record.id.startswith("GCF_000012325.1|"))
reference_columns = [index for index, residue in enumerate(str(reference.seq)) if residue != "-"]
property_rows = []
for record in records:
    sequence = "".join(str(record.seq)[index] for index in reference_columns if str(record.seq)[index] != "-")
    if any(symbol not in KD for symbol in sequence):
        raise SystemExit(f"nonstandard residue in {record.id}")
    counts = {aa: sequence.count(aa) for aa in KD}
    arg_lys = counts["R"] + counts["K"]
    property_rows.append({
        "assembly_accession": record.id.split("|", 1)[0],
        "proline_fraction": counts["P"] / len(sequence),
        "gravy": sum(KD[aa] for aa in sequence) / len(sequence),
        "arg_over_arg_plus_lys": counts["R"] / arg_lys if arg_lys else np.nan,
    })
properties = pd.DataFrame(property_rows)
meta = pd.read_csv("results/sequences/colap_clean_metadata.csv", dtype=str)
matched = pd.read_csv("results/comparison/matched_taxa.csv", dtype=str).fillna("")
ogt_table = pd.read_csv("data/gentilhomme/strain_ogt_github.tsv", sep="\t")
ogt_table["tokens"] = ogt_table["strn"].map(tokens)
growth = pd.read_csv("data/gentilhomme/growthrates.tsv", sep="\t")
growth["temp"] = pd.to_numeric(growth["temp"])
growth["tokens"] = growth["strain"].map(tokens)
rows = []
for record in meta.itertuples(index=False):
    if record.ogt_type == "absent":
        continue
    strain_tokens = tokens(f"{record.organism} {record.strain}")
    ogt_hits = ogt_table.loc[ogt_table["tokens"].map(lambda item: bool(item & strain_tokens))]
    growth_hits = growth.loc[growth["tokens"].map(lambda item: bool(item & strain_tokens))]
    species = matched.loc[(matched.colap_assembly == record.assembly_accession) & (matched.match_status == "matched")]
    temperatures = sorted(growth_hits["temp"].dropna().unique().tolist()) if len(growth_hits) else []
    ogt_value = float(record.ogt_value)
    gentilhomme_type = ogt_hits["Type"].iloc[0] if len(ogt_hits) == 1 else ""
    value_matches = bool(len(ogt_hits) == 1 and abs(float(ogt_hits["OGT"].iloc[0]) - ogt_value) < 1e-6)
    include = False
    reason = ""
    if len(ogt_hits) != 1:
        reason = "missing or ambiguous OGT match"
    elif not value_matches:
        reason = "OGT mismatch"
    elif gentilhomme_type != "ratk":
        reason = f"type {gentilhomme_type}"
    elif len(growth_hits) == 0:
        reason = "no rows in growthrates.tsv"
    elif len(species) != 1:
        reason = "no unique species-tree leaf"
    else:
        include = True
    rows.append({
        "assembly_accession": record.assembly_accession,
        "strain": record.strain,
        "organism": record.organism,
        "genus": record.genus,
        "ogt_celsius": ogt_value,
        "gentilhomme_type": gentilhomme_type,
        "measured_temperatures_celsius": ";".join(str(value) for value in temperatures),
        "patric_genome_id": "" if len(species) != 1 else species["patric_genome_id"].iloc[0],
        "include_in_primary": include,
        "exclusion_reason": reason,
    })
verified = pd.DataFrame(rows)
verified.to_csv(OUT / "ogt_verified.csv", index=False)
primary = verified[verified.include_in_primary].copy()
dataset = primary.merge(properties, on="assembly_accession", how="left")
if len(dataset) == 0 or dataset[PRIMARY].isna().any().any():
    raise SystemExit("missing ColAP properties for the primary OGT sample")
dataset.to_csv(OUT / "temperature_analysis_dataset.csv", index=False)
rename = dict(zip(primary.patric_genome_id, primary.assembly_accession))
species_tree = prune_to(Phylo.read("data/gentilhomme/species_tree.nwk", "newick"), set(rename))
for tip in species_tree.get_terminals():
    tip.name = rename[tip.name]
Phylo.write(species_tree, OUT / "species_tree_primary.nwk", "newick")
order = dataset.assembly_accession.tolist()
matrix, names = covariance(species_tree)
index = {name: i for i, name in enumerate(names)}
base = matrix[np.ix_([index[name] for name in order], [index[name] for name in order])]
y = dataset.ogt_celsius.to_numpy(dtype=float)
predictors = {name: dataset[name].to_numpy(dtype=float) for name in PRIMARY}
scales = {name: float(np.std(values, ddof=1)) for name, values in predictors.items()}
fits = {name: fit_lambda(y, predictors[name], base) for name in PRIMARY}
adjusted = holm([fits[name]["p"] for name in PRIMARY])
mean_fit = fit_lambda(y, None, base)
rows = []
for name, holm_p in zip(PRIMARY, adjusted):
    fit = fits[name]
    rows.append({
        "trait": name,
        "n": fit["n"],
        "beta_original_units": fit["slope"],
        "se_original_units": fit["se"],
        "ci95_low_original": fit["ci_low"],
        "ci95_high_original": fit["ci_high"],
        "beta_per_sd": fit["slope"] * scales[name],
        "ci95_low_per_sd": fit["ci_low"] * scales[name],
        "ci95_high_per_sd": fit["ci_high"] * scales[name],
        "predictor_sd": scales[name],
        "p": fit["p"],
        "p_holm": holm_p,
        "holm_family": "three_primary_traits",
        "lambda": fit["lambda"],
        "lambda_profile_ci95_low": fit["lambda_ci_low"],
        "lambda_profile_ci95_high": fit["lambda_ci_high"],
        "df": fit["df"],
        "ogt_phylogenetic_n_eff": effective_n(base, mean_fit["lambda"]),
    })
pd.DataFrame(rows).to_csv(OUT / "pgls_h1.csv", index=False)
